# Блокнот 7. Решающие деревья

Курс «Машинное обучение с нуля», модуль 7. Разделы: 1 — урок 7.2, 2 — урок 7.3, 3 — урок 7.4, 4 — урок 7.5. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_wine, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree

## 1. Энтропия и Джини (урок 7.2)

**Задание 1.** По числу объектов каждого класса верните энтропию в битах и критерий Джини. Нулевые доли в энтропии пропускайте.

In [ ]:
def impurity(counts):
    counts = np.asarray(counts, dtype=float)
    return ..., ...  # ваш код: (энтропия, джини)

In [ ]:
assert np.allclose(impurity([3, 1]), (0.811278, 0.375), atol=1e-5)
assert np.allclose(impurity([10, 0]), (0, 0))
assert np.allclose(impurity([1, 1, 1, 1]), (2, 0.75))
print("Верно!")

## 2. Корневой вопрос своими руками (урок 7.3)

**Задание 2.** Для одного признака найдите порог с наименьшим взвешенным Джини детей. Пороги — середины между соседними различными значениями.

In [ ]:
def best_split(x, y):
    order = np.argsort(x, kind="stable")
    xs, ys = x[order], y[order]
    n = len(xs)
    best_t, best_g = None, np.inf
    ...  # ваш код: перебор i, порог, взвешенный Джини, лучший вариант
    return best_t, best_g

Найдём лучший вопрос для корня на винах: переберём все 13 признаков и сравним с корнем дерева scikit-learn.

In [ ]:
data = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, stratify=data.target, random_state=0)
results = [(best_split(X_train[:, j], y_train), name) for j, name in enumerate(data.feature_names)]
(t, g), name = min(results, key=lambda r: r[0][1])
print(f"Лучший вопрос: {name} <= {t} (взвешенный Джини {g:.3f})")
root = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_train, y_train).tree_
assert data.feature_names[root.feature[0]] == name and np.isclose(root.threshold[0], t)
print("Совпадает с корнем DecisionTreeClassifier!")

Дерево регрессии не экстраполирует: обучим его на x от 0 до 6 и спросим про x до 9.

In [ ]:
rng = np.random.default_rng(3)
x = np.sort(rng.uniform(0, 6, 80))
y = x + rng.normal(0, 0.3, 80)  # прямая с шумом
reg = DecisionTreeRegressor(max_depth=3).fit(x[:, None], y)
grid = np.linspace(0, 9, 300)[:, None]
plt.scatter(x, y, s=10, color="gray")
plt.plot(grid, reg.predict(grid), label="дерево глубины 3")
plt.plot(grid, grid, "--", label="настоящая зависимость")
plt.axvline(6, color="red", lw=0.8)
plt.legend()
plt.title("Правее x = 6 дерево повторяет крайний лист")
plt.show()

## 3. Переобучение и обрезка (урок 7.4)

In [ ]:
Xm, ym = make_moons(600, noise=0.3, random_state=0)
Xtr, Xva, ytr, yva = train_test_split(Xm, ym, test_size=0.5, random_state=0)
depths = [1, 2, 3, 4, 5, 6, 8, 10, None]
train_acc, val_acc = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr)
    train_acc.append(m.score(Xtr, ytr))
    val_acc.append(m.score(Xva, yva))
labels = [str(d) for d in depths]
plt.plot(labels, train_acc, "o-", label="обучение")
plt.plot(labels, val_acc, "o-", label="валидация")
plt.xlabel("max_depth")
plt.legend()
plt.show()

**Задание 3.** Подберите `ccp_alpha` кросс-валидацией: возьмите все значения из `path.ccp_alphas`, для каждого посчитайте среднюю accuracy по 5 фолдам на `Xtr, ytr` и выберите лучшее.

In [ ]:
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(Xtr, ytr)
best_alpha = ...  # ваш код
pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=0).fit(Xtr, ytr)
print(f"α = {best_alpha:.4f}: листьев {pruned.get_n_leaves()}, валидация {pruned.score(Xva, yva):.3f}")
assert pruned.get_n_leaves() < DecisionTreeClassifier(random_state=0).fit(Xtr, ytr).get_n_leaves()

## 4. Дроид-сомелье (урок 7.5)

In [ ]:
small = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
plt.figure(figsize=(12, 5))
plot_tree(small, feature_names=data.feature_names, class_names=["сорт 0", "сорт 1", "сорт 2"], filled=True)
plt.show()
print(export_text(small, feature_names=data.feature_names))

In [ ]:
grid = GridSearchCV(DecisionTreeClassifier(random_state=0),
                    {"max_depth": [2, 3, 4, 5, None], "min_samples_leaf": [1, 3, 5, 10]}, cv=5).fit(X_train, y_train)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
print("Дерево:", grid.best_params_, "CV", round(grid.best_score_, 3), "тест", round(grid.score(X_test, y_test), 3))
print("Логистическая регрессия: CV", cross_val_score(logreg, X_train, y_train, cv=5).mean().round(3),
      "тест", round(logreg.fit(X_train, y_train).score(X_test, y_test), 3))
imp = sorted(zip(grid.best_estimator_.feature_importances_, data.feature_names), reverse=True)
for value, name in imp[:5]:
    print(f"{name:30s} {value:.3f}")